## Pre-processing + U-Net (training and testing)

#### Important notes training and testing!
Training this network with 5-fold cross-validation for 100 epochs usually takes 12-24 hours with a standard GPU. Using the Jupyter notebook server the time takes up to about 6 hours. 
When running with different amounts of epochs, look up the following line of code and change the name to the amount of epochs: name=f'unet-fold{fold_idx}-10epochs'. Furthermore, look out for the following during training:
1. Weights&Biases (W&B) is used to monitor both the training and testing of the network, for example for overfitting. Overfitting occurs when the model memorizes the training data, which results in a low trianing loss, but fails to generalize the valudation data, which results in a validation Dice score. To combat this, data augmentation and weight decay is applied. The weight decay will be explained later on.
2. The hyperparameters can be changed after running. The training now includes a learning rate scheduler. When larger amounts of epochs are used, this prevents the validation Dice from starting to jitter or plateau pre-maturely. The optimization algorithm ADAM also has an adaptive learning rate.
3. Do not forget to change the amounts of epochs before training and change the pathways if necessary. You probably only need to change the pathway for the training and test data, the rest it will do by itself. 
4. Necessary data gets displayed after running a section, for example the best training fold and Dice coefficient after training the model. Use the best fold for testing (fill in manually).

#### Package preparation, error fixing

It was necessary to get a Numpy version below 2.0, since it otherwise clashes with critical packages like MONAI. Furthermore, Pytorch Ignite must be installed.

In [ ]:
!pip install "numpy<2"
!pip install pytorch-ignite

#### Pre-processing - Fixed operations only

For the pre-processing, the following steps were included:
1. Read the volumes per patient. Since the training dataset contains 100 patients, a total of 200 volumes should be identified.
2. The pre-processing steps do not yet include transformations (data augmentation), which will be applied during training as has been taught in tutorial 3. This does, however, include shuffling the patients randomly using a specific seed number to ensure shuffling is the same each time. This simplifies training and testing the model on different PCs and/or laptops.
3. 5 folds are created to apply 5-fold validation during training. This means the 100 patients are split into groups of 20, which means each fold contains 40 volumes.
4. Pre-processed data is saved in a the respective map. The directory can be changed manually.

In [ ]:
# Import packages
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from tqdm import tqdm
import nibabel as nib
import pickle
import torch

from monai.transforms import ( 
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, 
    Resized, ScaleIntensityd, ToTensord
)
import SimpleITK as sitk
sitk.ProcessObject_SetGlobalWarningDisplay(False)

In [ ]:
# Import packages
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from tqdm import tqdm
import nibabel as nib
import pickle
import torch

from monai.transforms import ( 
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, 
    Resized, ScaleIntensityd, ToTensord
)
import SimpleITK as sitk
sitk.ProcessObject_SetGlobalWarningDisplay(False)

# 1. Create the data list
def prepare_acdc_data_list(data_root):
    data_list = []
    path = Path(data_root)
    patient_dirs = sorted([d for d in path.iterdir() if d.is_dir() and d.name.startswith('patient')])
    
    for patient_dir in patient_dirs:
        pid = patient_dir.name
        frame_files = sorted(list(patient_dir.glob('*_frame[0-1][0-9].nii.gz')))
        img_files = [f for f in frame_files if '_gt' not in f.name]
        
        for i, img_path in enumerate(img_files):
            mask_path = img_path.parent / img_path.name.replace('.nii.gz', '_gt.nii.gz')
            if mask_path.exists():
                data_list.append({
                    "image": str(img_path),
                    "label": str(mask_path),
                    "PatientID": pid,
                    "Phase": "ED" if i == 0 else "ES"
                })
    return data_list

# !! CHANGE DATAPATH !!
data_path = '/home/jovyan/Desktop/Project/Data/training'
raw_data_list = prepare_acdc_data_list(data_path)

# Volumes have to be read since it classifies every single voxel (3D pixel) into one of 4 categories: BG, RV, MYO, LV
print(f"Total volumes identified: {len(raw_data_list)}")

# 2. Create and save 5-fold patient splits -> preparation for 5-fold validation
unique_patients = np.unique([d["PatientID"] for d in raw_data_list])
# The seed ensures the patient shuffling is done the same everytime, regardless of who runs it
np.random.seed(42) 
np.random.shuffle(unique_patients)

kf = KFold(n_splits=5)
patient_to_fold = {p: f_nr for f_nr, (_, val_idx) in enumerate(kf.split(unique_patients), 1) 
                   for p in unique_patients[val_idx]}

# !! CHANGE PATHWAY !! to map where 5 folds are to be saved
# Output directory
output_root = Path("/home/jovyan/dlmia-course/Project/5_folds")
output_root.mkdir(parents=True, exist_ok=True)

with open(output_root.parent / "patient_to_fold.pkl", "wb") as f:
    pickle.dump(patient_to_fold, f)
print("Saved patient_to_fold.pkl mapping")

# 3. Pre-processing with no random transforms yet.
fixed_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    Resized(keys=["image", "label"], spatial_size=(256, 256, 64), mode=("bilinear", "nearest")),
    ScaleIntensityd(keys=["image"]),
    ToTensord(keys=["image", "label"])
])

# Apply pre-processing and save in respective folder
for item in tqdm(raw_data_list, desc="Preprocessing dataset"):
    patient_id = item["PatientID"]
    phase = item["Phase"]
    
    data = fixed_transforms(item)
    image = data["image"][0].numpy()
    label = data["label"][0].numpy()
    
    patient_folder = output_root / patient_id
    patient_folder.mkdir(exist_ok=True)
    
    nib.save(nib.Nifti1Image(image, np.eye(4)), patient_folder / f"{phase}_image.nii.gz")
    nib.save(nib.Nifti1Image(label, np.eye(4)), patient_folder / f"{phase}_label.nii.gz")

print(f"Preprocessing finished. Files saved to: {output_root}")

#### Combined 5-fold training and Weights&Biases logging
The pipeline for both training and testing has been done using 10 epochs. This is enough to detect errors but also ensures it does not take too long to figure out whether correct results are obtained. After the pipeline was completed and fully functioning, the amount of epochs was increased.

##### Hyperparameters and Experimental Setup
Based on iterative testing, the following configuration for the U-Net was finalized. Certain experiments with regards to the hyperparemeters and GPU constraints were conducted. The results of which including substantation can be seen below: 
1. Batch size: A batch size of 2 and 3 were compared. The batch size was initially set at 2 to prevent GPU memory constraints. However, running the code with a batch size of 3 was feasible, and since it provided more stable gradient updates while still staying within the GPU's memory limits.
2. Optimizer and learning rate: We utilized the ADAM optimizer. The reason we used an ADAM optimizer is because it combines stochastic gradient descent, with momentum and an adaptive learning rate, making the gradient more smooth. We tested with a learning rate of 1e-3 and 1e-5. The learning rate of 1e-5 proved to be too small for effective learning, thus a learning rate of 1e-3 was selected. To ensure even smoother convergence, this was paired with an extra learning rate scheduler: The Cosine Annealing Scheduler.
3. Loss function: Initially we only used a standard Dice Loss. However, once observing the first runs we noticed that the Dice Score was plateauing prematurely. Thus, the decision was made to switch to a hybrid DiceCELoss. This combines the Dice loss with cross-entropy, which looks at the propability or certainty of a each individual pixel being in the right class.
4. Regularization: To prevent overfitting, which means the model memorizes the training samples, a weight decay of 1e-5 was implemented. This ensures that the model maintains smaller weights. This will eventually lead to better generalization on unseen data, in this case the test set.
5. Validation: The decision was made to employ a 5-fold cross-validation. This ensures every patient was used for validation exactly once. This provides a robust estimate of the model's true performance across the entire dataset. Furthermore, one can select the best fold to implement for testing. 
6. Evaluation metrics: In addition to the Dice coefficient, which measures volumetric overlap, the 95th percentile Hausdorff distance (HD95) was added. This allows one to measure the boundary errors or distance errors in millimeters. The goal was to get this value below 10 mm.
7. Data augmentation: Data augmentation was applied as taught in tutorial 3 to artifically increase the diversity and size of the training set. This ensures that the model learns the actual anatomical structures rather than specific pixel coordinates. This reduces the risk of overfitting.
8. Epochs: 10 epochs were used to solely test the pipeline. These results were not used, however, for the final evaluation. The reason 100 epochs were chosen instead of 50 epochs for example comes down to convergence: Giving the model enough time to move from general learning to mastering the details. This was especially fundamental for the myocardium, which is quite a thin structure. Using less epochs decreased the Dice coefficient for this section, thus the decision was made to use 100 epochs.

In [ ]:
# Import necessary packages
import os
import pickle
import numpy as np
import torch
import wandb
from pathlib import Path
from tqdm import tqdm

from ignite.engine import Events, create_supervised_trainer, create_supervised_evaluator
from monai.data import Dataset, DataLoader, decollate_batch
from monai.handlers import MeanDice
from monai.handlers import HausdorffDistance
from monai.losses import DiceCELoss
from monai.networks.nets import UNet
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, RandRotated, 
    RandFlipd, ToTensord, Activations, AsDiscrete
)

# 1. Path configuration (adjust to local environment) 
# !! CHANGE PATH !! to path where pre-processed data is (5 folds)
preprocessed_root = Path("/home/jovyan/dlmia-course/Project/5_folds")

# 2. Load the 5-fold mapping saved during pre-processing
with open(preprocessed_root.parent / "patient_to_fold.pkl", "rb") as f:
    patient_to_fold = pickle.load(f)
    
# 3. Build Global datalist
data_dicts = []
for patient_folder in sorted(preprocessed_root.iterdir()):
    if not patient_folder.is_dir(): continue
    for phase in ["ED", "ES"]:
        img = patient_folder / f"{phase}_image.nii.gz"
        lbl = patient_folder / f"{phase}_label.nii.gz"
        if img.exists() and lbl.exists():
            data_dicts.append({
                "image": str(img), 
                "label": str(lbl), 
                "PatientID": patient_folder.name
            })
print(f"Total volumes available: {len(data_dicts)}")

# 4. Training hyperparameters

# !! CHANGE HYPERPARAMETERS !! if necessary.

## Re-assess after performing training once
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
max_epochs = 100  # Set your desired epoch count. Run with less when testing the pipeline
learning_rate = 1e-3
batch_size = 3

# 5. Helper function to prepare batch
def prepare_batch(batch, device=None, non_blocking=False):
    return (
        batch["image"].to(device), 
        batch["label"].to(device)
    )

overall_best_dice = -1
overall_best_fold = -1 

# 6. Start 5-fold loop - training
for fold_idx in range(1, 6):
    print(f"Fold {fold_idx}")    
    
    # Reset local best_dice for each new fold
    best_dice = -1
    
    # Split data based on fold mapping
    train_files = [d for d in data_dicts if patient_to_fold[d["PatientID"]] != fold_idx]
    val_files   = [d for d in data_dicts if patient_to_fold[d["PatientID"]] == fold_idx]
    print(f"Train volumes: {len(train_files)}, Validation volumes: {len(val_files)}")

    # Data loaders
    # Apply data augmentation as has been taught in tutorial 3
    train_ds = Dataset(data=train_files, transform=Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        RandRotated(keys=["image", "label"], range_x=0.2, prob=0.5, mode=("bilinear", "nearest")),
        RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.5),
        ToTensord(keys=["image", "label"])
    ]))
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(Dataset(data=val_files, transform=Compose([
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
        ToTensord(keys=["image", "label"])
    ])), batch_size=1, num_workers=2, pin_memory=True)

    # Initialize Weights&Biases for this run.
    wandb.init(
        project='ACDC-segmentation',
        name=f'unet-fold{fold_idx}-100epochs', # !! CHANGE !! When using other amount of epochs
        config={
            "fold": fold_idx,
            "architecture": "3D U-Net",
            "epochs": max_epochs,
            "learning_rate": learning_rate,
            "batch_size": batch_size
        }
    )

    # Initialize model, optimizer and loss
    # Model is a U-Net with 256 channels. 
    # U-Net from MONAI uses PReLU (parametric ReLU) -> Advanced version of ReLU
    net = UNet(
        spatial_dims=3, 
        in_channels=1, 
        out_channels=4, # Out channels consists of the 4 masks: BG, LV, MYO, RV
        channels=(16, 32, 64, 128, 256), 
        strides=(2, 2, 2, 2), 
        num_res_units=2
    ).to(device)
    
    # The optimizer algorithm is ADAM
    optimizer = torch.optim.Adam(net.parameters(), learning_rate)
    
    # Initialize learning rate schedular (especially necessary for larger amounts of epochs)
    # CosineAnnealingLR smoothly decreases the LR over the max_epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)
    
    # Loss function is Dice Loss. Dice loss looks at overlap between predicted region and ground truth. Making it better suited.
    # DiceCELoss combines Dice loss with cross-entropy to make it more stable.
    # SoftMax is used here solely at the end to turn numbers into probabolities (Ex. 98% chance that this section is LV, 2% chance that it is BG).
    loss_function = DiceCELoss(softmax=True, to_onehot_y=True, include_background=False)

    # Setup trainer and evaluater
    trainer = create_supervised_trainer(net, optimizer, loss_function, device=device, prepare_batch=prepare_batch)
    
    val_metrics = {
        "Mean_Dice": MeanDice(),
        "HD95": HausdorffDistance(percentile=95)
    }
    evaluator = create_supervised_evaluator(
        net, val_metrics, device=device, prepare_batch=prepare_batch,
        output_transform=lambda x, y, y_pred: (
            [AsDiscrete(argmax=True, to_onehot=4)(Activations(softmax=True)(i)) for i in decollate_batch(y_pred)],
            [AsDiscrete(to_onehot=4)(i) for i in decollate_batch(y)]
        )
    )
    
    @trainer.on(Events.ITERATION_COMPLETED(every=5))
    def log_training_loss(engine):
        wandb.log({"train/loss": engine.state.output})

    @trainer.on(Events.EPOCH_COMPLETED)
    def run_validation_and_save(engine):
        # Access global variables
        global best_dice, overall_best_dice, overall_best_fold 
        
        # A. Run validation
        # Use both mean Dice value and Handofer distance
        evaluator.run(val_loader)
        current_dice = evaluator.state.metrics["Mean_Dice"]
        current_hd95 = evaluator.state.metrics["HD95"]
        
        # B. Update learning rate
        # This must happen once per epoch
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        
        # C. Get a visual sample from the validation set
        sample_data = next(iter(val_loader))
        val_inputs = sample_data["image"].to(device)
        val_labels = sample_data["label"].to(device)
        
        net.eval()
        with torch.no_grad():
            val_outputs = net(val_inputs)
            # Convert probabilities to a single class mask (0, 1, 2, or 3)
            preds = torch.argmax(val_outputs, dim=1).detach().cpu().numpy()
            ground_truth = val_labels[0, 0].detach().cpu().numpy()
            
        # Select the middle slice of the 3D volume (e.g., slice 5 for a small volume)
        mid_slice = preds.shape[1] // 2 
        
        wandb.log({
            "segmentation_viz": wandb.Image(
                val_inputs[0, 0, mid_slice].cpu().numpy(), 
                masks={
                    "predictions": {"mask_data": preds[0, mid_slice], "class_labels": {0: "BG", 1: "RV", 2: "MYO", 3: "LV"}},
                    "ground_truth": {"mask_data": ground_truth[mid_slice], "class_labels": {0: "BG", 1: "RV", 2: "MYO", 3: "LV"}}
                }
            )
        })
        
        # D. Log to Weights&Biases
        wandb.log({
            "val/mean_dice": current_dice,
            "val/hd95": current_hd95,
            "train/lr": current_lr, 
            "epoch": engine.state.epoch
        })
        print(f"Epoch {engine.state.epoch} - Dice: {current_dice:.4f} - HD95: {current_hd95:.4f} - LR: {current_lr:.6f}")

        # E. Save the best model for THIS fold
        if current_dice > best_dice:
            best_dice = current_dice
            torch.save(net.state_dict(), f"best_model_fold{fold_idx}.pth")
            print(f"--- New best model for Fold {fold_idx} saved (Dice: {best_dice:.4f}) ---")
        
        # F. Track the best fold OVERALL!
        # Use this fold in the testing
        if current_dice > overall_best_dice:
            overall_best_dice = current_dice
            overall_best_fold = fold_idx

    # Start training this fold
    trainer.run(train_loader, max_epochs=max_epochs)
    wandb.finish()

# Final summary report
print(f"\n\n{'#'*50}")
print(f"Cross-validation summary")
print(f"{'#'*50}")
print(f"Overall Best Fold: {overall_best_fold}")
print(f"Highest Validation Dice: {overall_best_dice:.4f}")
print(f"Action: Use 'best_model_fold{overall_best_fold}.pth' for final testing.")
print(f"{'#'*50}")

#### Test data pre-processing (fixed operations)

To ensure valid inference and overlap between the pipeline for both training and testing, the test data is subjected to the exact same fixed preprocessing operations as the training and validation sets.

In the next section for the final evaluation, unlike the training phase, no data augmentation is applied to the test set. This ensure the model's performance is tested on the actual, unaltered patient scans.

Upon completion of the loop, the performance is quantified using the Dice coefficient and HD95, as explained before. These are logged in a results table, which are broken down by the respective anatomical structures.

In [ ]:
# Load necessary packages
import nibabel as nib
import numpy as np
from pathlib import Path
from tqdm import tqdm
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, 
    Resized, ScaleIntensityd, ToTensord
)

# Define the pathways
# !! CHANGE PATHWAY !! to pathway where your test data is
test_raw_path = Path("/home/jovyan/Desktop/Project/Data/testing")
test_output_root = Path("/home/jovyan/Desktop/Project/Processed_data_test")
test_output_root.mkdir(parents=True, exist_ok=True)

# Define same fixed transforms as in training cell
test_fixed_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    Resized(keys=["image", "label"], spatial_size=(256, 256, 64), mode=("bilinear", "nearest")),
    ScaleIntensityd(keys=["image"]),
    ToTensord(keys=["image", "label"])
])

# Identify raw test volumes 
test_files = []
for p_dir in sorted([d for d in test_raw_path.iterdir() if d.is_dir()]):
    images = sorted(list(p_dir.glob('patient*_frame[0-9][0-9].nii.gz')))
    for img_p in images:
        lbl_p = img_p.parent / img_p.name.replace('.nii.gz', '_gt.nii.gz')
        if lbl_p.exists():
            test_files.append({"image": str(img_p), "label": str(lbl_p), "ID": p_dir.name, "fname": img_p.stem})
            
# Execute pre-processing on test data
print(f"Pre-processing {len(test_files)} volumes...")
for item in tqdm(test_files, desc="Pre-processing test data"):
    data = test_fixed_transforms(item)
    
    # Extract numpy arrays for saving
    image_np = data["image"][0].numpy()
    label_np = data["label"][0].numpy()
    
    p_folder = test_output_root / item["ID"]
    p_folder.mkdir(exist_ok=True)
    
    # Save as NIfTI with consistent naming
    nib.save(nib.Nifti1Image(image_np, np.eye(4)), p_folder / f"{item['fname']}_img.nii.gz")
    nib.save(nib.Nifti1Image(label_np, np.eye(4)), p_folder / f"{item['fname']}_lbl.nii.gz")

print("Pre-processing complete.")

#### Testing & evaluation

In [ ]:
import pandas as pd
import torch
import wandb
import os
from pathlib import Path
from tqdm import tqdm
from monai.networks.nets import UNet
from monai.metrics import DiceMetric
from monai.metrics import HausdorffDistanceMetric
from monai.data import decollate_batch, Dataset, DataLoader
# Added missing imports below
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, 
    ToTensord, AsDiscrete, Activations
)

# 0. Setup device -> Check whether cuda is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Define the transforms
# Not necessary to perform pre-processing steps.
# Change to tensors
val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    ToTensord(keys=["image", "label"])
])

# Use to_onehot=4 to separate the 3 heart tissues + background correctly
post_pred = Compose([Activations(softmax=True), AsDiscrete(argmax=True, to_onehot=4)])
post_label = AsDiscrete(to_onehot=4)

# 2. Define pathway where processed data is present
test_processed_root = Path("/home/jovyan/Desktop/Project/Processed_data_test")

test_data_dicts = []
if test_processed_root.exists():    
    # Search for all processed patients in the folder
    for patient_folder in sorted(test_processed_root.iterdir()):
        if not patient_folder.is_dir(): 
            continue
        
        # Find all images (files ending in .nii.gz but NOT _gt)
        images = sorted(list(patient_folder.glob("*_img.nii.gz")))   
        for img_path in images:
            # Construct label path based on your pre-processing naming (_lbl.nii.gz)
            lbl_path = img_path.parent / img_path.name.replace("_img.nii.gz", "_lbl.nii.gz")   
            if lbl_path.exists():
                test_data_dicts.append({
                    "image": str(img_path), 
                    "label": str(lbl_path), 
                    "PatientID": patient_folder.name
                })        
    print(f"Total processed test volumes found: {len(test_data_dicts)}")
else:
    print(f"Error: Path {test_processed_root} not found. Run pre-processing first!")

# 3. Initialize evaluation metrics (one for the mean, one for per-class)
# Set include_background=False once more to focus on 3 heart structures.
test_dice_metric = DiceMetric(include_background=False, reduction="mean", get_not_nans=False)
test_dice_per_class = DiceMetric(include_background=False, reduction="mean_batch", get_not_nans=False)
test_hd95_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean", get_not_nans=False)

# 4. Load the single best fold!
# !! CHANGE THIS !! to the fold index (1-5) that performed best during training
# Result is displayed at training
best_fold_idx = 5 # Change this!
model_path = f"best_model_fold{best_fold_idx}.pth"

if not os.path.exists(model_path):
    print(f"Error: {model_path} not found!")
else:
    # Initialize and load weights
    test_net = UNet(
        spatial_dims=3, 
        in_channels=1, 
        out_channels=4,
        channels=(16, 32, 64, 128, 256), 
        strides=(2, 2, 2, 2), 
        num_res_units=2
    ).to(device)
    test_net.load_state_dict(torch.load(model_path, map_location=device))
    test_net.eval()

    # Create dataloader for 50 test patients
    test_ds = Dataset(data=test_data_dicts, transform=val_transforms)
    test_loader = DataLoader(test_ds, batch_size=1, num_workers=2)

    print(f"\nStarting testing: 50 patients (100 volumes) using fold {best_fold_idx}...")

    # 5. Inference Loop
    with torch.no_grad():
        for test_data in tqdm(test_loader):
            val_images, val_labels = test_data["image"].to(device), test_data["label"].to(device)
            val_outputs = test_net(val_images)
            
            val_outputs = [post_pred(i) for i in decollate_batch(val_outputs)]
            val_labels = [post_label(i) for i in decollate_batch(val_labels)]
            
            test_dice_metric(y_pred=val_outputs, y=val_labels)
            test_dice_per_class(y_pred=val_outputs, y=val_labels)
            test_hd95_metric(y_pred=val_outputs, y=val_labels)

    # 6. Final results 
    mean_dice = test_dice_metric.aggregate().item()
    class_dice = test_dice_per_class.aggregate().cpu().numpy()
    mean_hd95 = test_hd95_metric.aggregate().item() 

    # Create the final results table
    results_df = pd.DataFrame([{
        "Test Set": "Final Performance",
        "Mean Dice": f"{mean_dice:.4f}",
        "RV Dice": f"{class_dice[0]:.4f}",
        "MYO Dice": f"{class_dice[1]:.4f}",
        "LV Dice": f"{class_dice[2]:.4f}",
        "HD95 (mm)": f"{mean_hd95:.4f}"
    }])

    print("\n")
    print(results_df.to_string(index=False))

    # 7. Log to Weights & Biases
    wandb.init(project='ACDC-segmentation', name='Final_Test_Evaluation')
    
    # Get WandB table with all the results
    results_table = wandb.Table(dataframe=results_df)
    
    wandb.log({
        "test_mean_dice": mean_dice,
        "test_rv_dice": class_dice[0],
        "test_myo_dice": class_dice[1],
        "test_lv_dice": class_dice[2],
        "test_hd95": mean_hd95,
        "performance_table": results_table  # Logs the entire table
    })
    
    print("\nResults successfully logged to Weights & Biases.")
    wandb.finish()